# Hub and spoke, step 5: parallel research, then synthesis

Step 1 was one coordinator and one web search spoke. Step 2 added a second spoke and made both cite
their sources. Step 3 replaced their prose with a schema. Step 4 added the spoke that consumes it.
Step 5 stops running the research one spoke at a time.

| Spoke | Tools | Runs | Returns |
| :- | :- | :- | :- |
| `web-searcher` | `WebSearch` only | in parallel | `Finding[]` with `source_url` + `source_title` |
| `document-analyzer` | `Read`, `Grep`, `Glob` only | in parallel | `Finding[]` with `document_name` + `page_number` |
| `synthesis` | **none** | after both | the cited report, built from both |

Four ideas are doing the work.

**Tool restriction.** The searcher cannot open your files, the analyzer cannot reach the internet,
and synthesis can do neither. None of that is a prompt instruction you hope the model respects - a
tool left out of `AgentDefinition.tools` is not in that spoke's session at all.

**Structured provenance.** Each spoke runs in its own context and only its final message returns to
the hub, which never sees the search results or the file contents. Whatever provenance a spoke fails
to put in that message is gone. A `Finding` makes provenance a *field*, so it survives the trip.

**Lossless hand-off.** The synthesis spoke has no tools, so it can only cite what the hub passes it.
The hub is instructed to pass the complete findings array verbatim - and because an instruction is
not a guarantee, `check_handoff` diffs what it passed against what the spokes returned.

**Parallel where it is free.** The two research spokes never read each other's output, so running
them one after the other buys nothing and costs the slower one's latency twice. Issued in the same
response they overlap, and the chain is as slow as the slowest spoke rather than their sum.

In [42]:
# Setup: already satisfied in this repo's .venv.
#   claude-agent-sdk 0.2.177  -> in requirements.txt
#   Claude Code CLI 2.1.317   -> the SDK drives it under the hood
# Starting fresh elsewhere? Then:
# %pip install claude-agent-sdk
# (and install the CLI: https://code.claude.com/docs/en/agent-sdk/quickstart)

In [43]:
# Load env variables (ANTHROPIC_API_KEY) - same as the other notebooks
from pathlib import Path

from dotenv import load_dotenv

from claude_agent_sdk import (
    query,
    ClaudeAgentOptions,
    AgentDefinition,
    AssistantMessage,
    ResultMessage,
    TextBlock,
    ToolUseBlock,
    ToolResultBlock,
    UserMessage,
)

load_dotenv()

model = "claude-haiku-4-5"

# Run the agent from the repo root so file paths like "features/earth.pdf" resolve
# no matter which directory Jupyter was started in.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "agentic_loops" else Path.cwd()
print("repo root:", REPO_ROOT)

repo root: /Users/iCastro/Projects/study/anthropic/bca


## 1. The `Finding` schema: content vs. metadata

So far each spoke returned prose. Prose is fine for a human and bad for a synthesis agent: to cite
a claim it has to re-parse `(features/earth.pdf p. 4)` back out of a sentence, and anything the
spoke phrased loosely is simply lost.

A `Finding` fixes that by splitting every result in two:

- **`content`** - what was found: the `claim` and the `analysis` behind it. This is what gets read.
- **`metadata`** - where it came from: `source_url`, `document_name`, `page_number`, `confidence`,
  `retrieved_by`. This is what gets *cited*, and it never gets mixed into the prose.

That separation is the whole point. The synthesis agent writes its report from `content` and builds
its reference list from `metadata`, never having to guess which half of a sentence was the citation.

Three design decisions worth understanding, because they are where this kind of schema usually fails:

1. **Every metadata field is `required`, and the ones that do not apply must be explicitly `null`.**
   A web finding has no `page_number`; a document finding has no `source_url`. If those were merely
   optional, a spoke that *forgot* a page number would still produce valid output. Forcing an explicit
   `null` turns "I have no locator for this" into a statement the spoke makes, not a field it dropped.
2. **`anyOf` requires at least one real locator.** A finding with `source_url: null` *and*
   `document_name: null` cannot be cited by anyone, so the schema rejects it outright. That is the
   one guarantee the synthesis agent leans on.
3. **`confidence` is a three-value enum, not a 0-1 float.** Models are not calibrated well enough for
   `0.82` to mean anything stable; `high|medium|low` is a judgment they can make consistently.

In [44]:
# The Finding schema - one result from one spoke, content separated from metadata.
import json

from jsonschema import Draft202012Validator

FINDING_SCHEMA = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "title": "Finding",
    "description": "A single piece of evidence returned by a spoke agent.",
    "type": "object",
    "additionalProperties": False,
    "required": ["content", "metadata"],
    "properties": {
        # ---------------- CONTENT: what was found (this gets read) ----------------
        "content": {
            "type": "object",
            "additionalProperties": False,
            "required": ["claim", "analysis"],
            "properties": {
                "claim": {
                    "type": "string",
                    "minLength": 1,
                    "maxLength": 300,
                    "description": (
                        "One self-contained, verifiable sentence. No citation text inside it - "
                        "provenance belongs in metadata."
                    ),
                },
                "analysis": {
                    "type": "string",
                    "minLength": 1,
                    "description": (
                        "Why the claim matters, what supports it, and any caveat. 2-4 sentences. "
                        "May quote the source verbatim."
                    ),
                },
            },
        },
        # ------------- METADATA: where it came from (this gets cited) -------------
        "metadata": {
            "type": "object",
            "additionalProperties": False,
            # All of them required. Fields that do not apply must be an explicit null, so that
            # "no locator" is something the spoke states rather than something it forgot.
            "required": [
                "source_url",
                "source_title",
                "document_name",
                "page_number",
                "confidence",
                "retrieved_by",
            ],
            "properties": {
                "source_url": {
                    "type": ["string", "null"],
                    "format": "uri",
                    "description": "Full URL of the web source. null for local-document findings.",
                },
                "source_title": {
                    "type": ["string", "null"],
                    "description": (
                        "Exact title of the web page, so the report can cite something readable "
                        "instead of a bare link. null for local-document findings."
                    ),
                },
                "document_name": {
                    "type": ["string", "null"],
                    "description": (
                        "Repo-relative path of the local file, e.g. 'features/earth.pdf'. "
                        "null for web findings."
                    ),
                },
                "page_number": {
                    "type": ["integer", "null"],
                    "minimum": 1,
                    "description": (
                        "1-indexed page the claim came from. null for web findings and for "
                        "non-paginated files. A claim spanning pages becomes one Finding per page."
                    ),
                },
                "confidence": {
                    "type": "string",
                    "enum": ["high", "medium", "low"],
                    "description": (
                        "high: stated directly by the source. "
                        "medium: inferred from the source, or the source is weak. "
                        "low: indirect, contested, or one unconfirmed source."
                    ),
                },
                "retrieved_by": {
                    "type": "string",
                    "enum": ["web-searcher", "document-analyzer"],
                    "description": "Which spoke produced this finding. Must match the agent name.",
                },
            },
            # A finding nobody can cite is useless to the synthesis agent:
            # require at least ONE real locator (a URL or a document path).
            "anyOf": [
                {"properties": {"source_url": {"type": "string"}}, "required": ["source_url"]},
                {"properties": {"document_name": {"type": "string"}}, "required": ["document_name"]},
            ],
        },
    },
}

# What a spoke actually returns: an envelope holding a list of Findings.
FINDINGS_SCHEMA = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "title": "FindingsReport",
    "type": "object",
    "additionalProperties": False,
    "required": ["findings", "gaps"],
    "properties": {
        "findings": {"type": "array", "minItems": 1, "items": FINDING_SCHEMA},
        "gaps": {
            "type": "array",
            "items": {"type": "string"},
            "description": "What the spoke could NOT establish. Empty list if nothing is missing.",
        },
    },
}

# Compiled once, reused. check_schema catches a typo in the schema itself.
Draft202012Validator.check_schema(FINDINGS_SCHEMA)
findings_validator = Draft202012Validator(FINDINGS_SCHEMA)

# The compact rendering we paste into each spoke's prompt.
FINDINGS_SCHEMA_TEXT = json.dumps(FINDINGS_SCHEMA, indent=2)
print(f"schema ok - {len(FINDINGS_SCHEMA_TEXT)} chars to paste into each spoke prompt")

schema ok - 4406 chars to paste into each spoke prompt


### Validating a report against the contract

The Agent SDK has no per-subagent structured-output enforcement: a spoke's final message is plain
text and the schema lives in its prompt. So the schema is a contract you **check**, not one the
runtime guarantees - which is exactly why the check below matters.

`parse_findings` does both halves: pull JSON out of whatever the spoke wrote, then validate it. It
collects **every** error instead of raising on the first, because when a spoke drifts you want the
full list of what it got wrong in a form you can paste straight back into its prompt.

In [45]:
def extract_json_object(text):
    """Return the first balanced {...} or [...] in `text` that parses as JSON, else None.

    A spoke's report does not arrive alone: the Agent tool result also carries wrapper
    text such as an `agentId: <id>` trailer, so the JSON is rarely the whole string and
    may not be at character 0. Scanning for a balanced object is what makes the contract
    survive that, instead of failing with "Expecting value: line 1 column 1".
    """
    openers = {"{": "}", "[": "]"}
    for start, char in enumerate(text):
        if char not in openers:
            continue
        opener, closer = char, openers[char]
        depth, in_string, escaped = 0, False, False
        for end in range(start, len(text)):
            char = text[end]
            if in_string:
                if escaped:
                    escaped = False
                elif char == "\\":
                    escaped = True
                elif char == '"':
                    in_string = False
            elif char == '"':
                in_string = True
            elif char == opener:
                depth += 1
            elif char == closer:
                depth -= 1
                if depth == 0:
                    try:
                        return json.loads(text[start : end + 1])
                    except json.JSONDecodeError:
                        break  # not it - try the next opening bracket
    return None


def parse_findings(report_text):
    """Parse a spoke's final message into a validated findings report.

    Returns (report, errors). `report` is None when no JSON object could be found;
    `errors` is a list of readable strings, and empty means it validated.
    """
    report = extract_json_object(report_text)

    if report is None:
        # Show what the spoke actually said - without it you cannot tell a prose
        # answer apart from JSON buried in wrapper text.
        preview = " ".join(report_text.split())[:300]
        return None, [f"no JSON object found. The spoke returned: {preview!r}"]

    # iter_errors collects ALL problems; validate() would stop at the first.
    errors = []
    for err in sorted(findings_validator.iter_errors(report), key=lambda e: list(e.path)):
        location = "/".join(str(part) for part in err.path) or "(root)"
        errors.append(f"{location}: {err.message}")

    return report, errors


def citation(finding):
    """Render one Finding's metadata as a citation string."""
    meta = finding["metadata"]
    if meta.get("source_url"):
        return f'"{meta.get("source_title") or "untitled"}" - {meta["source_url"]}'
    page = f" p. {meta['page_number']}" if meta.get("page_number") else ""
    return f"{meta['document_name']}{page}"


# Smoke-test the contract on hand-written examples - no API call needed.
good = {
    "findings": [
        {
            "content": {
                "claim": "Earth's atmosphere is roughly 78% nitrogen.",
                "analysis": "Stated directly in the composition table, alongside 21% oxygen.",
            },
            "metadata": {
                "source_url": None,
                "source_title": None,
                "document_name": "features/earth.pdf",
                "page_number": 4,
                "confidence": "high",
                "retrieved_by": "document-analyzer",
            },
        }
    ],
    "gaps": [],
}

# Same finding, but with no locator at all - the anyOf rule must reject it.
uncitable = json.loads(json.dumps(good))
uncitable["findings"][0]["metadata"]["document_name"] = None

print("good      ->", parse_findings(json.dumps(good))[1] or "valid")
print("uncitable ->", parse_findings(json.dumps(uncitable))[1])
print("citation  ->", citation(good["findings"][0]))

good      -> valid
uncitable -> ["findings/0/metadata: {'source_url': None, 'source_title': None, 'document_name': None, 'page_number': 4, 'confidence': 'high', 'retrieved_by': 'document-analyzer'} is not valid under any of the given schemas"]
citation  -> features/earth.pdf p. 4


## 2. Spoke one: web search, citing title + URL

Two fields do the work in an `AgentDefinition`:

- `description` is read by the **hub** to decide when to delegate here. Write it for the hub.
- `tools` is the spoke's entire toolbox. `["WebSearch"]` means it can search and nothing else -
  no files, no bash.

The report format is in the prompt because that final message is all the hub gets. We ask for the
page **title** next to each URL: a bare link tells the hub nothing about what it is, and titles are
what make the hub's synthesized answer readable.

In [46]:
web_search_agent = AgentDefinition(
    # Read by the HUB to decide when to delegate here.
    description=(
        "Searches the web for one focused question and returns structured Findings, each carrying "
        "the title and URL of its source. Use for any question needing current or external information."
    ),
    prompt=(
        """You are a Web Search Specialist, a spoke in a hub-and-spoke system.

The Coordinator (the hub) gives you ONE focused question. You have no other context,
and your answer goes back to the hub only, never directly to the user.

How to work:
- Search the web before answering. Never answer from memory alone.
- Answer only the question you were given. Do not drift into neighboring topics.
- Never invent a fact, a title, or a URL. What you cannot verify goes in `gaps`.
- One claim per Finding. Split a compound statement into separate Findings.

YOUR ENTIRE FINAL MESSAGE MUST BE ONE JSON OBJECT matching this schema. The FIRST
character you write must be `{` and the LAST must be `}` - no sentence introducing it,
no summary after it, no markdown code fence, no apology if you found nothing (report
that in `gaps` instead):

"""
        + FINDINGS_SCHEMA_TEXT
        + """

Filling in the metadata, as the web-searcher:
- source_url:    the full URL you actually visited. REQUIRED for you - never null.
- source_title:  the exact page title, so the report can cite something readable.
- document_name: always null. You cannot read local files.
- page_number:   always null. Web pages have no pages.
- confidence:    high if the source states it directly, medium if you inferred it or the
                 source is weak, low if it is indirect or only one unconfirmed source says it.
- retrieved_by:  always the string "web-searcher".

Keep `claim` free of citation text - provenance belongs in metadata, and the synthesis
agent builds its reference list from there."""
    ),
    # Its entire toolbox: search only. No Read, no Bash - it cannot touch the filesystem.
    tools=["WebSearch"],    # Foreground: a spawned subagent runs in the BACKGROUND by default and the Agent tool
    # returns status "async_launched", with the report arriving some later turn. The two
    # research spokes are issued together and both are needed before synthesis, so pin
    # them to blocking - the hub gets the finished report, not an acknowledgement.
    background=False,
)

## 3. Spoke two: document analysis, citing page references

Same shape, opposite capabilities. `["Read", "Grep", "Glob"]` is the standard **read-only** set:
`Read` opens a file (and takes a `pages` argument for PDFs), `Glob` finds files by name, `Grep`
searches inside them. No `Write`, no `Edit`, no `Bash`, and no `WebSearch` - this spoke physically
cannot modify anything or go online.

For locators we ask for `p. N` on PDFs and `path:line` on text files, since that is what the
underlying tools actually hand back: `Read` on a PDF works in pages, while `Grep` returns line numbers.
Asking for a citation format the tools can't produce is how you get invented ones.

In [47]:
document_analysis_agent = AgentDefinition(
    # Read by the HUB to decide when to delegate here.
    description=(
        "Analyzes local documents and returns structured Findings, each carrying the file path and "
        "page number it came from. Use for any question about a file in this repository."
    ),
    prompt=(
        """You are a Document Analyst, a spoke in a hub-and-spoke system.

The Coordinator (the hub) gives you a file (or a place to look) and an analysis task.
You have no other context, and your analysis goes back to the hub only, never directly
to the user.

How to work:
- Analyze ONLY what you read from the files. Add no outside knowledge; you cannot search
  the web and must not pretend to.
- Use Glob to locate files and Grep to find passages, then Read the relevant parts.
- For a PDF, read it in SMALL page ranges - at most 5 pages per Read call - so you always
  know which page a claim came from and never pull the whole file into one message.
  Each PDF page comes back as an image, so whole-file reads are slow and wasteful.
- Never invent a page number or a quote. A claim you cannot place on a page goes in `gaps`.
- One claim per Finding. A claim spanning two pages becomes two Findings.

YOUR ENTIRE FINAL MESSAGE MUST BE ONE JSON OBJECT matching this schema. The FIRST
character you write must be `{` and the LAST must be `}` - no sentence introducing it,
no summary after it, no markdown code fence, no apology if you found nothing (report
that in `gaps` instead):

"""
        + FINDINGS_SCHEMA_TEXT
        + """

Filling in the metadata, as the document-analyzer:
- source_url:    always null. You cannot search the web.
- source_title:  always null.
- document_name: the repo-relative path you read, e.g. "features/earth.pdf". REQUIRED for
                 you - never null.
- page_number:   the 1-indexed page the claim came from, or null for a file with no pages
                 (use the `analysis` field to note the line number instead).
- confidence:    high if the document states it directly, medium if you inferred it from
                 the text, low if the document only hints at it.
- retrieved_by:  always the string "document-analyzer".

Put short verbatim quotes in `analysis`, and keep `claim` free of citation text -
provenance belongs in metadata, and the synthesis agent cites from there."""
    ),
    # Read-only toolbox: open, find, and search files. No Write/Edit/Bash, no WebSearch.
    tools=["Read", "Grep", "Glob"],    # Foreground: a spawned subagent runs in the BACKGROUND by default and the Agent tool
    # returns status "async_launched", with the report arriving some later turn. The two
    # research spokes are issued together and both are needed before synthesis, so pin
    # them to blocking - the hub gets the finished report, not an acknowledgement.
    background=False,
)

## 4. The hub's output contract: enforced, not requested

Everything so far has been a contract we *ask* for and then *check*. `output_format` is different:
it is the one place in this pipeline where the schema is enforced **server-side**. The hub cannot
return prose that violates it, and the result arrives on `ResultMessage.structured_output` as a dict.

Note the asymmetry, because it shapes everything else:

| | Spokes | Hub |
| :- | :- | :- |
| Channel | final message (free text) | `output_format` |
| Enforcement | our `findings_validator`, after the fact | the API, before you ever see it |
| On violation | you get bad data and must retry | you cannot get bad data |

`AgentDefinition` has no `output_format` field, so this applies to the hub only - which is exactly
the failure you hit: a spoke returned prose instead of Findings and nothing could stop it. The hub's
answer is now the one link in the chain that cannot drift.

**The catch: `output_format` accepts a smaller JSON Schema dialect than your local validator.**
`minLength`, `maxLength` and `minimum` are rejected outright - sending `FINDING_SCHEMA` as-is would
400 - and `minItems` accepts only 0 or 1. So we sanitize: strip what the API rejects, and let the
local validator keep the strict version. One source of truth, two strictness levels.

In [48]:
# The hub's final deliverable: a cited report. It reuses the Finding metadata fields, so a
# reference carries exactly the provenance a spoke collected - nothing is retyped by hand.
import copy

# Keywords the API rejects inside an output_format schema (they cause a 400). The local
# jsonschema validator keeps them; only the server-facing copy is sanitized.
_UNSUPPORTED = ("minLength", "maxLength", "minimum", "maximum", "multipleOf", "$schema")


def strip_for_output_format(schema):
    """Return a copy of `schema` using only the dialect output_format accepts."""
    if isinstance(schema, list):
        return [strip_for_output_format(item) for item in schema]
    if not isinstance(schema, dict):
        return schema

    clean = {}
    for key, value in schema.items():
        if key in _UNSUPPORTED:
            continue
        clean[key] = min(value, 1) if key == "minItems" else strip_for_output_format(value)
    return clean


# A reference = the Finding's metadata, plus an id to cite it by and the claim it supports.
_reference_props = copy.deepcopy(FINDING_SCHEMA["properties"]["metadata"]["properties"])
_reference_props["id"] = {
    "type": "string",
    "description": "Short marker used inside the report text, e.g. 'R1'.",
}
_reference_props["claim"] = {
    "type": "string",
    "description": "The claim this reference supports, copied from the spoke's Finding.",
}

CITED_REPORT_SCHEMA = strip_for_output_format(
    {
        "type": "object",
        "additionalProperties": False,
        "required": ["summary", "report", "references", "gaps"],
        "properties": {
            "summary": {
                "type": "string",
                "description": "One or two sentences answering the question directly.",
            },
            "report": {
                "type": "string",
                "description": (
                    "The full prose answer. Every factual sentence ends with its reference "
                    "marker in square brackets, e.g. '... roughly 78% nitrogen [R1].'"
                ),
            },
            "references": {
                "type": "array",
                "minItems": 1,
                "items": {
                    "type": "object",
                    "additionalProperties": False,
                    "required": list(_reference_props),
                    "properties": _reference_props,
                },
                "description": "Every reference marker used in `report`, in order.",
            },
            "gaps": {
                "type": "array",
                "items": {"type": "string"},
                "description": "What could not be established. Empty list if nothing is missing.",
            },
        },
    }
)

Draft202012Validator.check_schema(CITED_REPORT_SCHEMA)
report_validator = Draft202012Validator(CITED_REPORT_SCHEMA)

print("reference fields:", sorted(_reference_props))
print("stripped before sending:", [k for k in _UNSUPPORTED if f'"{k}"' in json.dumps(FINDING_SCHEMA)])

reference fields: ['claim', 'confidence', 'document_name', 'id', 'page_number', 'retrieved_by', 'source_title', 'source_url']
stripped before sending: ['minLength', 'maxLength', 'minimum', '$schema']


In [49]:
def check_report(report):
    """Post-checks the server cannot make: markers resolve, references are citable.

    Normalizes enum casing IN PLACE first. The API may return an enum value
    recapitalized ("High" for "high"), which is valid output but would fail a
    case-sensitive local check, so normalize before validating rather than after.
    """
    for ref in report.get("references", []):
        if isinstance(ref.get("confidence"), str):
            ref["confidence"] = ref["confidence"].lower()

    problems = [
        f"{'/'.join(str(x) for x in e.path) or '(root)'}: {e.message}"
        for e in report_validator.iter_errors(report)
    ]

    for ref in report.get("references", []):
        # The at-least-one-locator rule (anyOf) lives in FINDING_SCHEMA but is not part of
        # the server-facing copy, so it is checked here instead.
        if not ref.get("source_url") and not ref.get("document_name"):
            problems.append(f"reference {ref.get('id')}: no locator - it cannot be cited")
        if ref.get("id") and f"[{ref['id']}]" not in report.get("report", ""):
            problems.append(f"reference {ref['id']}: never cited in the report text")

    return problems


def render(report):
    """Print a structured report the way a reader wants it."""
    print(report["summary"])
    print()
    print(report["report"])
    print("\nReferences:")
    for ref in report["references"]:
        # citation() takes a Finding, and a reference carries the same metadata fields.
        print(f"  [{ref['id']}] {citation({'metadata': ref})}  ({ref['confidence']})")
    for gap in report.get("gaps", []):
        print(f"  [gap] {gap}")

## 5. Spoke three: synthesis, with zero tools

The synthesis spoke writes the cited report. It is the first spoke with `tools=[]` - **no** tools at
all - and that is deliberate: it cannot search, cannot open a file, cannot verify anything. It can
only use what the hub puts in its prompt.

That constraint is what makes this step interesting. Every earlier spoke could go get what it
needed; this one is entirely at the mercy of the hand-off. If the hub paraphrases a claim, drops a
`page_number`, or "tidies up" a URL on the way in, the synthesis spoke has no way to notice and no
way to recover - it will cite whatever it was given, confidently.

So the hub's instruction is now specific: pass the findings array **verbatim**, as JSON, with every
metadata field intact. Not the claim text, not a summary - the whole structure.

> **Be clear about what this guarantees: nothing, by itself.** The hub is a language model copying
> JSON from its context into a tool call, and models paraphrase, truncate, and "helpfully" drop
> fields that look redundant. The instruction raises the odds; it does not enforce anything.
> That is why the next cell adds `check_handoff`, which reads the delegation prompt the hub actually
> wrote and diffs it against the findings the spokes actually returned. If a field was lost in
> transit, you will see which one, on which claim.

In [50]:
synthesis_agent = AgentDefinition(
    # Read by the HUB to decide when to delegate here.
    description=(
        "Writes the final cited report from a complete findings array. Use after the research "
        "spokes have returned their Findings. Requires the full structured findings, not a summary."
    ),
    prompt=(
        """You are the Synthesis Specialist, a spoke in a hub-and-spoke system.

The Coordinator (the hub) gives you a COMPLETE findings array collected by the other
spokes, as JSON. You have NO TOOLS: you cannot search, open files, or verify anything.
Everything you are allowed to assert must already be present in those findings.

How to work:
- Write from `content.claim` and `content.analysis`. Cite from `metadata`.
- Use EVERY finding you were given, or say in `gaps` why one was left out.
- Never add a fact that no finding supports, and never invent or "fix" a URL, title or
  page number. If a finding's metadata looks incomplete, report that in `gaps` - do not
  fill it in from your own knowledge, which may be wrong or out of date.
- When two findings disagree, say so in the report rather than silently picking one.
- Weigh `confidence`: lead with high-confidence claims, and attribute low-confidence ones
  explicitly ("one unconfirmed report says ...").
- Preserve `retrieved_by` so the report shows which spoke produced each reference.

Give every reference a short id (R1, R2, ...), end each factual sentence in `report` with
its marker - "... roughly 78% nitrogen [R1]." - and copy each reference's metadata fields
across UNCHANGED from the finding it came from.

YOUR ENTIRE FINAL MESSAGE MUST BE ONE JSON OBJECT matching this schema. The FIRST
character you write must be `{` and the LAST must be `}` - no sentence introducing it,
no summary after it, no markdown code fence:

"""
        + json.dumps(CITED_REPORT_SCHEMA, indent=2)
    ),
    # No tools at all. This spoke reasons over the hand-off and nothing else.
    tools=[],
)

In [51]:
# The metadata fields that must survive the hand-off intact.
METADATA_FIELDS = tuple(FINDING_SCHEMA["properties"]["metadata"]["properties"])


def check_handoff(captured, delegation_prompt):
    """Diff what the spokes returned against what the hub passed to synthesis.

    `captured` is the list of (spoke, report) pairs collected so far. Returns a list of
    problems: findings dropped on the way in, and metadata fields altered or removed.
    Everything the hub typed is visible in the Agent tool input, so this is checkable.
    """
    expected = [f for _, report in captured for f in report["findings"]]
    if not expected:
        return ["nothing captured from the spokes, so the hand-off cannot be verified"]

    passed = extract_json_object(delegation_prompt)
    if passed is None:
        return [f"no findings JSON in the synthesis prompt ({len(delegation_prompt)} chars of prose?)"]

    # Accept either a bare array or the {"findings": [...]} envelope.
    if isinstance(passed, dict):
        passed = passed.get("findings", [])
    by_claim = {f.get("content", {}).get("claim"): f for f in passed if isinstance(f, dict)}

    problems = []
    for finding in expected:
        claim = finding["content"]["claim"]
        got = by_claim.get(claim)
        if got is None:
            problems.append(f"DROPPED or reworded: {claim[:70]!r}")
            continue
        for field in METADATA_FIELDS:
            want, have = finding["metadata"][field], got.get("metadata", {}).get(field, "<missing>")
            if want != have:
                problems.append(f"{field} changed on {claim[:45]!r}: {want!r} -> {have!r}")

    extra = len(by_claim) - len(expected)
    if extra > 0:
        problems.append(f"{extra} finding(s) in the prompt that no spoke returned - invented?")

    return problems

## 6. The hub: one coordinator, three spokes

Registering a spoke is one more entry in `agents`. With three of them, the things to notice:

- `allowed_tools` grows to cover **everything the spokes use**. It is an auto-approval list
  ("run these without asking"), not a restriction list - and a notebook cannot answer a permission
  prompt. Restriction is done per-spoke in `tools`.
- The hub's prompt now has a **routing rule** and an **ordering rule**. `description` alone decides
  where work goes (web question → searcher, file question → analyzer), but synthesis must run
  *last*, after the research spokes have reported - a spoke with no tools has nothing to work with
  until then.
- The hand-off instruction is the longest rule in the prompt, and deliberately so: it is the only
  thing standing between a complete findings array and a summarised one.
- `setting_sources=[]` keeps this self-contained: no `CLAUDE.md` or `.claude/` settings from this
  repo enter the agent's context.

In [52]:
coordinator_prompt = """You are the Coordinator, the central hub in a hub-and-spoke system.

You orchestrate; you do not answer from your own knowledge. Three spokes are available:

- `web-searcher`      - searches the web. Use it for current or external information.
- `document-analyzer` - reads local files. Use it for anything about a file in this repository.
- `synthesis`         - writes the final cited report. It has NO TOOLS and knows nothing except
                        what you pass it, so it runs LAST, after the research spokes report back.

Rules:
1. Route by capability. A question about the world goes to `web-searcher`; a question about a
   file goes to `document-analyzer`. Never answer either kind yourself.
2. Delegate the research IN PARALLEL. When a question needs both research spokes, issue BOTH
   Agent calls in the SAME response - two tool calls in one turn. They are independent, so
   never wait for one report before requesting the other, and never split them across turns.
   Then wait for BOTH reports before doing anything else.
3. Each spoke starts with a FRESH context and sees ONLY the prompt you pass it. Make every
   delegation self-contained: state the question, the scope, the file path if there is one,
   and what to report back. Two parallel delegations are two separate self-contained prompts.
4. Hand off to `synthesis` only once BOTH research spokes have reported. In that delegation
   prompt you MUST include the COMPLETE findings array from every spoke, verbatim, as JSON:

       {"findings": [ <every Finding object exactly as the spoke returned it> ], "gaps": [...]}

   Copy each object whole - content.claim, content.analysis, and ALL SIX metadata fields
   (source_url, source_title, document_name, page_number, confidence, retrieved_by), nulls
   included. Do NOT summarize, shorten, reword, merge or drop findings, and do not pass only
   the claim text. The synthesis spoke has no tools and cannot recover anything you leave out:
   a field you omit is a citation the final report can never make.
5. Preserve citations end to end. Every reference you emit must come from a spoke's Finding
   metadata, copied across unchanged. Never invent a reference, and never cite something no
   spoke returned.
6. Return the synthesis spoke's report as your answer, structured. Give every reference a short
   id (R1, R2, ...), end each factual sentence in `report` with its marker - "... 78% nitrogen
   [R1]." - and list every marker in `references`. Anything that could not be established goes
   in `gaps` rather than being stated without a citation."""
options = ClaudeAgentOptions(
    model=model,
    system_prompt=coordinator_prompt,
    cwd=str(REPO_ROOT),
    # "Agent" is the delegation tool (formerly "Task"); both listed for version safety.
    # The rest are auto-approved because the spokes use them and a notebook can't answer a prompt.
    allowed_tools=["Agent", "Task", "WebSearch", "Read", "Grep", "Glob"],
    # The two spokes, registered by name.
    agents={
        "web-searcher": web_search_agent,
        "document-analyzer": document_analysis_agent,
        "synthesis": synthesis_agent,
    },
    # Ignore this repo's CLAUDE.md / .claude settings so the example stays self-contained.
    setting_sources=[],
    # Reading a PDF page returns it as a base64 IMAGE, and one such tool result can blow past
    # the SDK's default 1 MB stdout buffer (CLIJSONDecodeError: "exceeded maximum buffer size").
    # 32 MB is plenty for page images; this is a transport limit, not an API one.
    max_buffer_size=32 * 1024 * 1024,
    # The one server-side guarantee in this pipeline: the hub's final answer MUST match
    # this schema. It arrives on ResultMessage.structured_output as a dict, already parsed.
    output_format={"type": "json_schema", "schema": CITED_REPORT_SCHEMA},
)

## 7. A helper to run the hub and watch the delegation

`query()` streams messages. We print three things so the pattern stays visible:

- `Agent` / `Task` tool calls → the hub delegating, and to which spoke
- messages carrying `parent_tool_use_id` → work happening **inside** a spoke's context
- the final `ResultMessage` → the hub's synthesized answer, and the cost of the whole run

In [53]:
import time


def _result_text(block):
    """A ToolResultBlock's content is a str or a list of content dicts."""
    if isinstance(block.content, str):
        return block.content
    return "\n".join(
        part.get("text", "") for part in (block.content or []) if isinstance(part, dict)
    )


async def run_hub(prompt):
    """Run one hub turn. Prints who does what, and validates every spoke report.

    Returns (validated_spoke_reports, hub_report). The hub report is a dict whose
    shape the API guaranteed; the spoke reports are whatever survived local validation.
    """
    delegations = {}  # tool_use_id -> spoke name, so we know which results are spoke reports
    issued_at = {}    # tool_use_id -> when the delegation went out, to measure overlap
    spoke_seconds = 0.0
    started = time.monotonic()
    validated = []    # spoke reports that passed our local validator
    final = None      # the hub's structured report

    async for message in query(prompt=prompt, options=options):
        # Is this message coming from INSIDE a spoke's context?
        where = "  [spoke]" if getattr(message, "parent_tool_use_id", None) else "[hub]  "

        if isinstance(message, AssistantMessage):
            # Two Agent calls in ONE assistant message is what parallel delegation looks
            # like on the wire. Spread across two messages, they would run one after the other.
            same_turn = [
                b for b in message.content
                if isinstance(b, ToolUseBlock) and b.name in ("Task", "Agent")
            ]
            if len(same_turn) > 1:
                names = ", ".join(str(b.input.get("subagent_type")) for b in same_turn)
                print(f"{where} PARALLEL: {len(same_turn)} delegations in one response -> {names}")

            for block in message.content:
                # Match BOTH names: current SDKs emit "Agent", older ones emitted "Task".
                if isinstance(block, ToolUseBlock) and block.name in ("Task", "Agent"):
                    spoke = block.input.get("subagent_type")
                    handoff = str(block.input.get("prompt") or "")
                    delegations[block.id] = spoke
                    issued_at[block.id] = time.monotonic()
                    print(f"{where} DELEGATES -> {spoke}")
                    print(f"{where}   prompt: {handoff[:200]}...")

                    # The synthesis spoke has no tools, so whatever the hub failed to copy
                    # into this prompt is simply gone. Diff it against what we captured.
                    if spoke == "synthesis":
                        losses = check_handoff(validated, handoff)
                        if losses:
                            print(f"{where}   !! hand-off lost data:")
                            for loss in losses[:10]:
                                print(f"{where}        {loss}")
                        else:
                            kept = sum(len(r["findings"]) for _, r in validated)
                            print(f"{where}   ok: all {kept} finding(s) passed through intact")
                elif isinstance(block, ToolUseBlock):
                    print(f"{where} tool: {block.name} {str(block.input)[:120]}")
                elif isinstance(block, TextBlock) and block.text.strip():
                    print(f"{where} {block.text.strip()[:300]}")

        # Tool results arrive as user messages. The ones whose id we delegated are
        # spoke reports - the moment the structured Findings cross back to the hub.
        elif isinstance(message, UserMessage):
            blocks = message.content if isinstance(message.content, list) else []
            for block in blocks:
                if isinstance(block, ToolResultBlock) and block.tool_use_id in delegations:
                    spoke = delegations[block.tool_use_id]
                    took = time.monotonic() - issued_at.get(block.tool_use_id, started)
                    spoke_seconds += took
                    print(f"\n   {spoke} came back after {took:.1f}s")
                    raw = _result_text(block)

                    # The synthesis spoke returns a cited REPORT, not Findings, so it is
                    # held to a different contract than the research spokes.
                    if spoke == "synthesis":
                        drafted = extract_json_object(raw)
                        if drafted is None:
                            print(f"\n!! {spoke} returned no JSON report")
                            print(f"     raw ({len(raw)} chars): {' '.join(raw.split())[:300]!r}")
                        else:
                            problems = check_report(drafted)
                            label = "drafted a report" if not problems else "report has problems"
                            print(f"\n>> {spoke} {label} "
                                  f"({len(drafted.get('references', []))} reference(s))")
                            for problem in problems[:8]:
                                print(f"     {problem}")
                        continue

                    report, errors = parse_findings(raw)
                    if errors:
                        print(f"\n!! {spoke} report FAILED validation:")
                        for err in errors[:8]:
                            print(f"     {err}")
                        # The raw text is the only way to tell "returned prose" apart
                        # from "returned JSON wrapped in harness text".
                        print(f"     raw result ({len(raw)} chars): {' '.join(raw.split())[:400]!r}")
                    else:
                        validated.append((spoke, report))
                        print(f"\n>> {spoke} returned {len(report['findings'])} valid finding(s):")
                        for finding in report["findings"]:
                            meta = finding["metadata"]
                            print(f"     [{meta['confidence']:>6}] {finding['content']['claim']}")
                            print(f"              cite: {citation(finding)}")
                        for gap in report["gaps"]:
                            print(f"     [  gap ] {gap}")

        elif isinstance(message, ResultMessage):
            print("\n=== FINAL ANSWER (from the hub) ===")
            report = message.structured_output

            if report is None:
                # No structured output: the run errored, or output_format did not apply.
                print("(no structured output - falling back to raw text)")
                print(message.result)
            else:
                # Shape is guaranteed by the API. What it cannot check is whether the
                # markers resolve and every reference is actually citable.
                problems = check_report(report)
                if problems:
                    print("!! report passed the schema but failed post-checks:")
                    for problem in problems:
                        print(f"     {problem}")
                    print()
                render(report)
                final = report

            elapsed = time.monotonic() - started
            print(f"\ncost: ${message.total_cost_usd}")
            # If the spokes' combined time exceeds the wall clock, they overlapped.
            print(f"wall clock: {elapsed:.1f}s | spokes combined: {spoke_seconds:.1f}s", end=" ")
            print("-> they overlapped" if spoke_seconds > elapsed else "-> no overlap (sequential)")

    return validated, final

### Run the full chain

`features/earth.pdf` is a snapshot of Wikipedia's *Earth* article taken on 2025-05-17, and page 3
carries exact figures: 0.0415% atmospheric CO2, a 14.76 °C mean surface temperature. That makes a
question no single spoke can answer:

1. **`document-analyzer`** reads the PDF for what the snapshot says, cited by page, **and**
2. **`web-searcher`** looks up what those figures are now, cited by title and URL -
   both issued in the *same* coordinator response, so they run at the same time.
3. **`synthesis`** then receives both sets of Findings, metadata intact, and writes one report.

Two mechanisms make that happen, and you need both:

- **The prompt** tells the hub to issue both Agent calls in one response and wait for both reports.
- **`background=False`** on each research spoke pins it to blocking. This matters because a spawned
  subagent runs in the **background** by default: the Agent tool returns `status: "async_launched"`
  and the report lands some later turn. Parallel-but-unawaited is not what we want here - synthesis
  must not start on half the findings.

Parallelism is only safe because these two spokes are independent. Synthesis stays sequential by
necessity: it has no tools, so it has nothing to do until the hand-off arrives.

In [54]:
reports, final = await run_hub(
    "features/earth.pdf is a snapshot of Wikipedia's Earth article taken on 2025-05-17. "
    "First, have the document analyst extract from that PDF what it states for atmospheric "
    "carbon dioxide concentration and for mean surface temperature, with the page number for "
    "each. Then have the web searcher find the most recent published values for the same two "
    "measurements. Finally, have the synthesis agent write one short report comparing the "
    "snapshot with the current figures and saying how much each has changed."
)

# Both locator kinds in one reference list: p. N from the PDF, URLs from the web.
print("\nreferences as data:")
for ref in final["references"]:
    print(f"  {ref['retrieved_by']:>18} | {ref['confidence']:>6} | {citation({'metadata': ref})}")

print(f"\nspoke findings captured: {sum(len(r['findings']) for _, r in reports)}")

⚠ claude.ai connectors are disabled because ANTHROPIC_API_KEY or another auth source is set and takes precedence over your claude.ai login · Unset it to load your organization's connectors


[hub]   I'll start with the document analyst on the PDF.
[hub]   DELEGATES -> document-analyzer
[hub]     prompt: Read the local file features/earth.pdf (repo-relative path; it is a snapshot of Wikipedia's "Earth" article taken 2025-05-17). Extract exactly what it states for:
1. Atmospheric carbon dioxide (CO2) c...
  [spoke] tool: Glob {'pattern': '**/earth.pdf', 'path': '/Users/iCastro/Projects/study/anthropic/bca'}
  [spoke] tool: Read {'file_path': '/Users/iCastro/Projects/study/anthropic/bca/features/earth.pdf', 'pages': '1-5'}
  [spoke] tool: Grep {'pattern': 'ppm|carbon dioxide|°C|K\\)', 'path': '/Users/iCastro/Projects/study/anthropic/bca/features/earth.pdf', 'out
  [spoke] tool: Grep {'pattern': 'ppm|carbon dioxide|°C|K\\)', 'path': '/Users/iCastro/Projects/study/anthropic/bca/features/earth.pdf', 'out
  [spoke] tool: Grep {'pattern': '/Type\\s*/Page[^s]|FlateDecode', 'path': '/Users/iCastro/Projects/study/anthropic/bca/features/earth.pdf', 
  [spoke] tool: Read {'file_path': 

### What to look for

The run should now show **two** delegations in one turn, then a third:

```
[hub]   PARALLEL: 2 delegations in one response -> document-analyzer, web-searcher
   document-analyzer came back after 12.4s
   web-searcher came back after 18.1s
[hub]   DELEGATES -> synthesis
[hub]     ok: all 4 finding(s) passed through intact
...
wall clock: 24.3s | spokes combined: 36.8s -> they overlapped
```

That last line is the measurement, not a claim: when the spokes' combined time exceeds the wall
clock, they genuinely ran at the same time. If you see `-> no overlap (sequential)` and one
`DELEGATES` line per turn, the hub ignored rule 2 and answered in sequence - the instruction is
steering, not enforcement, and this is how you find out.

Then compare the four contracts, which fail in visibly different ways:

| | Spokes | Hand-off | Parallelism | Hub |
| :- | :- | :- | :- | :- |
| Mechanism | JSON in the prompt | copy verbatim | one turn, `background=False` | `output_format` |
| Checked by | `findings_validator` | `check_handoff` | the timing line | the API |
| When it breaks | `!! FAILED validation` | `!! hand-off lost data` | `no overlap` | cannot break |

Only the last is enforcement. The other three are detection - which is the honest state of this
pipeline, and worth seeing rather than assuming away.

Also worth noticing:

- The final reference list mixes `p. 3` with URLs, and `retrieved_by` shows which spoke produced
  each one. That is why `Finding` was one type with nullable locators instead of two shapes.
- The snapshot is from May 2025, so the CO2 and temperature figures should differ from today's. A
  synthesis report that calls them identical did not really read both sets of findings.
- Parallelism has limits set outside the prompt: `CLAUDE_CODE_MAX_CONCURRENT_SUBAGENTS` (default 20)
  caps how many run at once, and `CLAUDE_CODE_MAX_SUBAGENT_SPAWN_DEPTH` (default 3) caps how deeply
  they nest. Both are set through the `env` option, and `max_budget_usd` caps the spend.

Next step, if you keep going: make the spoke channel enforced rather than checked - a
`submit_findings` MCP tool, so structured data is the only channel a spoke has.